# 09 · Vehicle Energy Dataset: monitored by configuration only

Phase 9, Step 9.6. The second dataset through the whole pipeline, with no code written for it
(Step 9.5): its months' health, what its rules found, its incidents and their explanations, a
real change the monitor found, and the fault experiments on it.

**Before running:**
```
python src/data_download/fetch_archives.py --dataset ved_signals
python src/ingestion/ingest_data.py --dataset ved_signals
python src/gold/rebuild.py --dataset ved_signals --lookups data/incoming
python src/evaluation/run_experiments.py --config configs/faults_ved.yml   # section 6
```
The last section prints a verdict to paste back.

In [ ]:
import json
import os
import sys
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(REPO_ROOT / "src"))

from common import db
from common.config import load_datasets_config
from evaluation import evaluate as ev
from evaluation import explanation_accuracy as xa
from evaluation import fault_injector as fi
from silver.process_silver import current_silver

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 110)
DATASET = "ved_signals"
INK, BLUE, SURFACE = "#52514e", "#3987e5", "#fcfcfb"
HEALTH = {"healthy": "#0ca30c", "warning": "#fab219", "critical": "#d03b3b", "not_scored": "#8a8a85"}
OUTCOME = {"top1": "#0ca30c", "top3": "#fab219", "wrong": "#d03b3b", "unknown": "#8a8a85",
           "not_detected": "#d9d8d4"}
DATABASE = Path(os.environ.get("DATAGUARD_DATABASE", load_datasets_config().paths["duckdb"]))
con = db.connect(DATABASE, read_only=True)

## 1 · Health per month

In [ ]:
health = con.execute(
    "SELECT period, bronze_rows AS rows, quarantine_rows, score, health, checks_failed, "
    "incidents_active FROM gold_dataset_health WHERE dataset = ? "
    "ORDER BY period", [DATASET]
).df()
fig = go.Figure()
for label, colour in HEALTH.items():
    part = health[health["health"] == label]
    if not part.empty:
        fig.add_bar(x=part["period"], y=part["score"], name=label, marker_color=colour,
                    text=part["score"].round(1), textposition="outside", customdata=part["rows"],
                    hovertemplate="%{x}: score %{y:.1f}, %{customdata:,} rows<extra></extra>")
fig.update_layout(title="Quality score per month", yaxis=dict(range=[0, 112]),
                  plot_bgcolor=SURFACE, bargap=0.25, legend_title_text="health")
fig.show()
health

## 2 · What the rules found

In [ ]:
rules = con.execute(
    """
    WITH latest AS (
        SELECT * FROM rule_run_log WHERE dataset = ?
        QUALIFY row_number() OVER (PARTITION BY rule_id, period ORDER BY run_id DESC) = 1
    )
    SELECT rule_id, action, sum(rows_failed) AS rows_failed,
           round(100 * max(rows_failed / rows_checked), 3) AS worst_month_pct
    FROM latest GROUP BY ALL ORDER BY rows_failed DESC, rule_id
    """, [DATASET]
).df()
rules

## 3 · Incidents and their likely causes

In [ ]:
incidents = con.execute(
    """
    SELECT i.incident_id, i.title, i.severity, i.status, i.first_seen_period, i.last_seen_period,
           v.top_cause, v.confidence, v.where_verdict
    FROM incidents i LEFT JOIN investigations v USING (incident_id)
    WHERE i.dataset = ? ORDER BY i.incident_id
    """, [DATASET]
).df()
incidents

## 4 · A real change: petrol cars that stop sending fuel trims

The NULL-rate incidents on the fuel trims. Per month: the petrol cars (ICE) that send no long
term fuel trim at all, and those that send it in every reading.

In [ ]:
files = [p.as_posix() for pair in current_silver(con, DATASET).values() for p in pair]
trims = con.execute(
    f"""
    WITH v AS (
        SELECT strftime(row_time, '%Y-%m') AS period, vehicle_id,
               avg((ltft_bank1_pct IS NULL)::INT) AS missing
        FROM read_parquet({files!r}, union_by_name = true, hive_partitioning = false)
        WHERE engine_type = 'ICE' GROUP BY ALL
    )
    SELECT period, count(*) AS petrol_cars, count_if(missing > 0.95) AS no_trims,
           count_if(missing < 0.05) AS all_trims
    FROM v GROUP BY 1 ORDER BY 1
    """
).df() if files else pd.DataFrame(columns=["period", "petrol_cars", "no_trims", "all_trims"])
fig = go.Figure(go.Bar(x=trims["period"], y=trims["no_trims"], marker_color=BLUE,
                       hovertemplate="%{x}: %{y} petrol cars without fuel trims<extra></extra>"))
fig.update_layout(title="Petrol cars sending no fuel trim at all", yaxis_title="cars",
                  plot_bgcolor=SURFACE, bargap=0.25)
fig.show()
trims

## 5 · The where/when limit

The investigation compares the affected rows with all rows of the month. Here they come from
dozens of cars at once and electric cars never send fuel trims, so no single segment holds
most of them: the answer is "no part of the data standing out" (docs/ved.md).

In [ ]:
incidents.loc[incidents["title"].str.contains("NULL rate"), ["incident_id", "title", "where_verdict"]]

## 6 · Fault experiments on VED (configs/faults_ved.yml)

In [ ]:
faults_path = os.environ.get("DATAGUARD_FAULTS_CONFIG", str(REPO_ROOT / "configs" / "faults_ved.yml"))
faults = fi.load_faults_config(Path(faults_path))
try:
    detected = ev.evaluate(faults)
    judged = xa.evaluate_explanations(faults)
    accuracy = xa.accuracy(judged)
    alarms = ev.control_findings(faults)
except FileNotFoundError as exc:
    detected = judged = accuracy = alarms = None
    print(f"no experiments yet: {exc}")
if detected is not None:
    counts = detected["outcome"].value_counts().to_dict()
    print(f"detection: {counts}; the control month raises: {alarms or 'nothing'}")
    print(json.dumps(accuracy, indent=2))
    display(judged[["experiment_id", "detector", "top_cause", "outcome", "where_claimed",
                    "where_outcome"]])

In [ ]:
if judged is not None and not judged.empty:
    table = judged.groupby(["fault_id", "outcome"]).size().unstack(fill_value=0)
    fig = go.Figure()
    for outcome in xa.OUTCOMES:
        if outcome in table:
            fig.add_bar(y=table.index, x=table[outcome], name=outcome, orientation="h",
                        marker_color=OUTCOME[outcome],
                        hovertemplate="%{y}: %{x} experiment(s) " + outcome + "<extra></extra>")
    fig.update_layout(barmode="stack", title="Explanations per fault on VED",
                      xaxis=dict(title="experiments", dtick=1, tickformat="d"),
                      yaxis_autorange="reversed", legend_traceorder="normal",
                      plot_bgcolor=SURFACE, height=max(260, 34 * len(table) + 120))
    fig.show()

## 7 · Verdict\n\nCopy the printed text back.

In [ ]:
verdict = {
    "months": len(health),
    "rows": int(health["rows"].sum()),
    "health": health["health"].value_counts().to_dict(),
    "rules_quarantining": rules.loc[(rules["action"] == "quarantine") & (rules["rows_failed"] > 0),
                                    "rule_id"].tolist(),
    "incidents": incidents[["incident_id", "title", "top_cause"]].to_dict("records"),
    "petrol_cars_without_trims": dict(zip(trims["period"], trims["no_trims"].astype(int))),
}
if detected is not None:
    verdict["detection"] = detected["outcome"].value_counts().to_dict()
    verdict["control_findings"] = alarms
    verdict["explanations"] = {k: accuracy[k] for k in ("detected", "top1", "top3", "wrong",
                                                        "unknown", "false_where", "where_found")}
print(json.dumps(verdict, indent=2, default=str))

In [ ]:
con.close()